In [ ]:
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt

# mock data (can also select lower tol one)
hdulist_mock = fits.open('./dla_mock_match.fits')
data_mock = hdulist_mock[1].data

z_dla_mock = data_mock['Z_DLA_GP']
targetid_mock = data_mock['TARGETID']

combined_mask = ((data_mock['IS_DLA_GP'] == 1) &
    (data_mock['SNR_REDSIDE'] > 1.0) & (data_mock['NHI_GP'] > 20.3))


z_dla_mock = data_mock['Z_DLA_GP'][combined_mask]
targetid_mock = data_mock['TARGETID'][combined_mask]

# true catalog
hdulist_true = fits.open('./dla_cat.fits')
data_true = hdulist_true[1].data

true_mask = data_true['NHI'] > 20.3
z_dla_true = data_true['Z_DLA'][true_mask]
targetid_true = data_true['TARGETID'][true_mask]

z_dla_true = data_true['Z_DLA']
targetid_true = data_true['TARGETID']

print(f"True z range: {np.min(z_dla_true)} - {np.max(z_dla_true)}")
print(f"Mock z range: {np.min(z_dla_mock)} - {np.max(z_dla_mock)}")

# run this function per LOS:
def match_dlas(true_z, mock_z, dz_tol=0.01):
    true_z = list(true_z)
    mock_z = list(mock_z)

    # matched_true = set()
    # matched_mock = set()
    matched_true = []
    matched_mock = []
    pairs = []

    for i, zt in enumerate(true_z):
        for j, zm in enumerate(mock_z):
            dz = abs(zt - zm)
            if dz < dz_tol:
                pairs.append((dz, i, j))

    pairs.sort()

    for pair in pairs:
        dz, i, j = pair

        if i in matched_true or j in matched_mock:
            continue

        matched_true.append(i)
        matched_mock.append(j)

    return matched_true, matched_mock

# compute number of true positives
# compute number of all dla detections in mock
recovered = np.zeros(len(z_dla_true), dtype=bool)
det_TP = np.zeros(len(z_dla_mock), dtype=bool)

all_ids = np.unique(targetid_mock)

for i, tid in enumerate(all_ids):
    if(i % 1000 == 0):
        print("Iteration " + str(i))
        
    true_mask = targetid_true == tid
    mock_mask = targetid_mock == tid

    true_z = z_dla_true[true_mask]
    mock_z = z_dla_mock[mock_mask]

    # target id not found in set
    if len(true_z) == 0 and len(mock_z) == 0:
        continue

    matched_true, matched_mock = match_dlas(true_z, mock_z, dz_tol=0.01)

    true_indices = np.where(true_mask)[0]
    mock_indices = np.where(mock_mask)[0]

    for j in matched_true: # use this for completeness
        recovered[true_indices[j]] = True

    for k in matched_mock: # use this for purity
        det_TP[mock_indices[k]] = True



In [ ]:
# create z-bins:
Nz = 20
z_min = min(np.min(z_dla_true), np.min(z_dla_mock))
z_max = max(np.max(z_dla_true), np.max(z_dla_mock))
z_bins = np.linspace(z_min, z_max, Nz + 1)
z_centers = 0.5 * (z_bins[:-1] + z_bins[1:])


# compute completeness per bin (using true z_dla)
completeness = []

for i in range(Nz):

    z_min = z_bins[i]
    z_max = z_bins[i + 1]
    in_current_bin = (z_dla_true >= z_min) & (z_dla_true < z_max) # act as a mask

    n_true = np.sum(in_current_bin)
    n_rec = np.sum(recovered[in_current_bin])
    print(str(n_rec) + " over " +  str(n_true))

    # if n_true > 0:
    #     c = n_rec / n_true
    # else:
    #     c = np.nan

    c = n_rec / n_true

    completeness.append(c)



# compute purity per bin (using mock z_dla)
purity = []
n_det_all = []

for i in range(Nz):
    
    z_min = z_bins[i]
    z_max = z_bins[i + 1]
    in_current_bin = (z_dla_mock >= z_min) & (z_dla_mock < z_max)

    n_det = np.sum(in_current_bin)
    n_det_all.append(n_det)
    n_tp = np.sum(det_TP[in_current_bin])

    # if n_det > 0:
    #     p = n_tp / n_det
    # else:
    #     p = np.nan

    p = n_tp / n_det
    purity.append(p)


# Plot c and p:

plt.scatter(z_centers, completeness, label='Completeness', c='Red')
plt.scatter(z_centers, purity, label='Purity', c='Blue')
plt.xlabel('Z_DLA')
plt.ylabel('Fraction')
plt.ylim(0, 1)
plt.legend()
plt.title('DLA Finder Performance vs Redshift')

plt.show()

In [ ]:

plt.scatter(z_centers, n_det_all)